# Exercise 12: Feature Engineering on Rover Survey Data

## Lab 12 (completed)
1. **Simplify** — `CONVERT_XY` Python UDF to map easting/northing onto tile / survey unit / subcell labels
2. **Aggregate** — multi-level averages at subcell, survey unit, and tile grain
3. **Assess** — classify tile averages against Ra-226 reference ranges (OK / Warning / Alarm)
4. **Rank / Score** — use Snowflake Cortex AI to suggest remediation priorities

## Exercise 12 additional features
Selected from the assignment feature list:
- **Hotspot Flag** (Simplify) — binary flag for tiles whose max reading crosses the Alarm threshold
- **Sensor Variability Index** (Combine) — coefficient of variation per tile (stddev / mean)
- **Z-Score vs Global Mean** (Assess) — how far each tile sits from the full-dataset distribution

All objects are created in `DATA5035.GIRAFFE`.

In [0]:
%%sql -r source_data
-- Work in my own schema
USE SCHEMA DATA5035.GIRAFFE;

SELECT * FROM DATA5035.SPRING26.SDG_001_RA226_SCANDATA LIMIT 10;

## Convert from Coordinates to Grid

The input data is provided in directional distances on a flat map projection. Northing and Easting indicate how far to go in those directions (up and right) in US Feet relative to a known starting point. Our purpose here is to map those directional distances on to a three-level grid.

### Measurement
* **Tiles** are the largest areas. They are composed of a grid of **Survey Units** 21 tiles wide x 18 tiles tall.
* **Survey Units** measure 32.81 ft x 32.81 ft square
* **Subcells** are square subdivisions within the **Survey Units** laid out 10x10

### Labeling
* **Tiles** are coded by row letter and column letter starting at AA in the bottom-left of our map, given some known origin (2180160.001, 6660000.000). AA indicates 1st row, 1st column. AB indicates 1st row, 2nd column to the left. BA indicates 2nd row up, 1st column.
* Within each Tile, **Survey Units** are numbered starting in the top-left corner, proceeeding right, then down to the beginning of the next row (as if you're reading down a page)
* Within each Survey Unit, **Subcells** are numbered starting in the bottom-left corner, proceeduing right, then up to the beginning of thext row (as if you're reading from the bottom of a page up)

**Create a Python UDF to convert x (easting), y (northing) into the grid labels.**

### Testing

```
    >>> convert_xy(2180160.0001, 6660000.0000)  
    ('AA', 358, 1)

    >>> convert_xy(2180160.0001 + 32.81*21.01, 6660000.0000)
    ('AB', 358, 1)

    >>> convert_xy(2180160.0001 + 32.81*22.01 + 4, 6660000.0000 + 32.81*19.01 + 4)
    ('BB', 338, 12)
```

In [0]:
%%sql -r dataframe_2
CREATE OR REPLACE FUNCTION CONVERT_XY(
        X FLOAT,
        Y FLOAT,
        ORIGIN_X FLOAT,
        ORIGIN_Y FLOAT,
        SU_SIZE FLOAT,
        TILE_GRID_X NUMBER(38,0),
        TILE_GRID_Y NUMBER(38,0),
        SUBCELL_GRID NUMBER(38,0)
    )
    RETURNS OBJECT
    LANGUAGE PYTHON
    RUNTIME_VERSION = '3.11'
    HANDLER = 'convert_xy'
    AS 
    $$
def convert_xy(x, y, origin_x, origin_y, su_size, tile_grid_x, tile_grid_y, subcell_grid):
    # Distance from origin measured in survey units (floating)
    su_x = (x - origin_x) / su_size
    su_y = (y - origin_y) / su_size

    # Which tile: columns increase going right (A, B, C...), rows increase going up (A, B, C...)
    tile_col_idx = int(su_x // tile_grid_x)
    tile_row_idx = int(su_y // tile_grid_y)
    tile = chr(ord('A') + tile_row_idx) + chr(ord('A') + tile_col_idx)

    # Position within the current tile, still in SU units
    su_x_in_tile = su_x - tile_col_idx * tile_grid_x
    su_y_in_tile = su_y - tile_row_idx * tile_grid_y

    # Survey units are numbered starting top-left, reading left-to-right then top-to-bottom
    su_col = int(su_x_in_tile)
    su_row_from_top = (tile_grid_y - 1) - int(su_y_in_tile)
    su = su_row_from_top * tile_grid_x + su_col + 1

    # Subcells are numbered starting bottom-left, reading left-to-right then bottom-to-top
    subcell_size = su_size / subcell_grid
    x_into_su = (su_x_in_tile - int(su_x_in_tile)) * su_size
    y_into_su = (su_y_in_tile - int(su_y_in_tile)) * su_size
    sub_col = int(x_into_su // subcell_size)
    sub_row_from_bottom = int(y_into_su // subcell_size)
    subcell = sub_row_from_bottom * subcell_grid + sub_col + 1

    return {'tile': tile, 'su': su, 'subcell': subcell}
    $$;

In [0]:
%%sql -r dataframe_3
CREATE OR REPLACE FUNCTION CONVERT_XY(X FLOAT, Y FLOAT)
RETURNS OBJECT
LANGUAGE SQL
AS
$$
    SELECT CONVERT_XY(
        X,
        Y,
        2180160.0001::NUMBER(20,10)::FLOAT,  -- origin easting
        6660000.0000::NUMBER(20,10)::FLOAT,  -- origin northing
        32.8100000000::NUMBER(20,10)::FLOAT, -- survey unit size (ft)
        21,                                   -- tile width in SUs
        18,                                   -- tile height in SUs
        10                                    -- subcell grid per SU
    )
$$;

In [0]:
%%sql -r dataframe_4
select convert_xy(2180160.0001, 6660000.0000);

In [0]:
%%sql -r dataframe_5
select convert_xy(2180160.0001 + 32.81*21.01, 6660000.0000);

In [0]:
%%sql -r dataframe_6
select convert_xy(2180160.0001 + 32.81*22.01 + 4, 6660000.0000 + 32.81*19.01 + 4)

In [0]:
%%sql -r dataframe_7
SELECT 
    convert_xy(easting, northing) AS coordinates, 
    coordinates:su::INTEGER AS su,
    coordinates:subcell::INTEGER AS subcell,
    coordinates:tile::STRING AS tile,
    * 
FROM 
    data5035.spring26.sdg_001_ra226_scandata 
LIMIT 100;

## Aggregate: Layered Averages

Build a base view with the tile / SU / subcell labels resolved, then aggregate up the three levels:
subcell → survey unit → tile. Each level gets count, mean, stddev, min, and max so downstream
features (hotspot flag, variability, z-score) can reuse it.

In [0]:
%%sql -r dataframe_8
-- Base view: resolve tile/su/subcell for every reading
CREATE OR REPLACE VIEW SCAN_LABELED AS
SELECT
    CONVERT_XY(easting, northing) AS coords,
    coords:tile::STRING     AS tile,
    coords:su::INTEGER      AS su,
    coords:subcell::INTEGER AS subcell,
    easting,
    northing,
    reading
FROM DATA5035.SPRING26.SDG_001_RA226_SCANDATA;

-- Subcell-level aggregates (finest grain)
CREATE OR REPLACE VIEW SCAN_SUBCELL_AGG AS
SELECT
    tile,
    su,
    subcell,
    COUNT(*)            AS n_readings,
    AVG(reading)   AS avg_reading,
    STDDEV(reading) AS stddev_reading,
    MIN(reading)   AS min_reading,
    MAX(reading)   AS max_reading
FROM SCAN_LABELED
GROUP BY tile, su, subcell;

-- Survey-unit-level aggregates
CREATE OR REPLACE VIEW SCAN_SU_AGG AS
SELECT
    tile,
    su,
    COUNT(*)            AS n_readings,
    AVG(reading)   AS avg_reading,
    STDDEV(reading) AS stddev_reading,
    MIN(reading)   AS min_reading,
    MAX(reading)   AS max_reading
FROM SCAN_LABELED
GROUP BY tile, su;

-- Tile-level aggregates (coarsest grain, drives most downstream features)
CREATE OR REPLACE VIEW SCAN_TILE_AGG AS
SELECT
    tile,
    COUNT(*)            AS n_readings,
    AVG(reading)   AS avg_reading,
    STDDEV(reading) AS stddev_reading,
    MIN(reading)   AS min_reading,
    MAX(reading)   AS max_reading
FROM SCAN_LABELED
GROUP BY tile;

SELECT * FROM SCAN_TILE_AGG ORDER BY avg_reading DESC;

## Compare to Reference Ranges

This measurement is of Radium-226 levels.
* `<5` - OK
* `5 <= X < 7.4` - Warning
* `>= 7.4` - Alarm

In [0]:
%%sql -r dataframe_9
-- Classify tile-level average against Ra-226 reference ranges
CREATE OR REPLACE VIEW SCAN_TILE_STATUS AS
SELECT
    tile,
    n_readings,
    avg_reading,
    max_reading,
    CASE
        WHEN avg_reading < 5    THEN 'OK'
        WHEN avg_reading < 7.4  THEN 'Warning'
        ELSE 'Alarm'
    END AS avg_status,
    CASE
        WHEN max_reading < 5    THEN 'OK'
        WHEN max_reading < 7.4  THEN 'Warning'
        ELSE 'Alarm'
    END AS max_status
FROM SCAN_TILE_AGG;

SELECT avg_status, COUNT(*) AS tile_count
FROM SCAN_TILE_STATUS
GROUP BY avg_status
ORDER BY CASE avg_status WHEN 'Alarm' THEN 1 WHEN 'Warning' THEN 2 ELSE 3 END;

## Prioritize

Once we've build everything out, let's use AI to make some recommendations on remediation priorities.

In [0]:
%%sql -r dataframe_10
-- Use Cortex to generate a remediation recommendation per Alarm/Warning tile.
-- We pass a compact summary of each concerning tile and ask for a priority + rationale.
CREATE OR REPLACE VIEW SCAN_TILE_PRIORITY AS
SELECT
    t.tile,
    t.avg_status,
    t.max_status,
    t.avg_reading,
    t.max_reading,
    t.n_readings,
    SNOWFLAKE.CORTEX.COMPLETE(
        'claude-4-sonnet',
        'You are a radiation remediation analyst. Given these Ra-226 survey readings for a tile, '
        || 'recommend a remediation priority (Immediate / High / Medium / Low / None) and give a '
        || 'one-sentence rationale. Ra-226 reference ranges: <5 OK, 5-7.4 Warning, >=7.4 Alarm. '
        || 'Tile: ' || t.tile
        || ', avg reading: ' || ROUND(t.avg_reading, 3)
        || ', max reading: ' || ROUND(t.max_reading, 3)
        || ', number of readings: ' || t.n_readings
        || '. Respond in the format: PRIORITY - rationale.'
    ) AS ai_recommendation
FROM SCAN_TILE_STATUS t
WHERE t.avg_status <> 'OK' OR t.max_status = 'Alarm';

SELECT * FROM SCAN_TILE_PRIORITY ORDER BY avg_reading DESC;

---

# Exercise 12: Additional Engineered Features

Three features chosen from the assignment catalog. Each is built on top of `SCAN_TILE_AGG`
so the logic composes cleanly.

| # | Feature | Category | Why it's useful |
|---|---------|----------|-----------------|
| 1 | Hotspot Flag | Simplify | Binary alarm indicator — drops the decision down to yes/no for operational triage |
| 2 | Sensor Variability Index | Combine | Normalizes noise by level, so a messy low-reading tile doesn't look as urgent as a messy high-reading one |
| 3 | Z-Score vs Global Mean | Assess | Puts each tile on a common scale — how unusual is it relative to the whole survey? |

## Feature 1: Hotspot Flag (Simplify)

Flag = 1 when a tile's max reading crosses the Alarm threshold (7.4). Keeps the signal binary
so it can be joined onto a map or used as a filter without re-applying thresholds everywhere.

In [0]:
%%sql -r dataframe_1
CREATE OR REPLACE VIEW FEATURE_HOTSPOT_FLAG AS
SELECT
    tile,
    max_reading,
    CASE WHEN max_reading >= 7.4 THEN 1 ELSE 0 END AS hotspot_flag
FROM SCAN_TILE_AGG;

SELECT hotspot_flag, COUNT(*) AS tile_count
FROM FEATURE_HOTSPOT_FLAG
GROUP BY hotspot_flag;

## Feature 2: Sensor Variability Index (Combine)

Coefficient of variation: `stddev / avg`. Combines two raw aggregates into a single normalized
number. Useful because a tile averaging 2.0 with stddev 1.0 is very different from a tile
averaging 8.0 with stddev 1.0 — same stddev, wildly different reliability.

In [0]:
%%sql -r dataframe_11
CREATE OR REPLACE VIEW FEATURE_VARIABILITY_INDEX AS
SELECT
    tile,
    avg_reading,
    stddev_reading,
    stddev_reading / NULLIF(avg_reading, 0) AS variability_index
FROM SCAN_TILE_AGG;

SELECT * FROM FEATURE_VARIABILITY_INDEX
ORDER BY variability_index DESC NULLS LAST
LIMIT 20;

## Feature 3: Z-Score vs Global Mean (Assess)

`(tile_avg - global_avg) / global_stddev`. Places every tile on the same standardized scale
relative to the full survey. A z-score above ~2 flags a statistical outlier worth investigating
regardless of the absolute threshold.

In [0]:
%%sql -r dataframe_12
CREATE OR REPLACE VIEW FEATURE_TILE_ZSCORE AS
WITH global_stats AS (
    SELECT
        AVG(reading)    AS global_avg,
        STDDEV(reading) AS global_stddev
    FROM SCAN_LABELED
)
SELECT
    t.tile,
    t.avg_reading,
    g.global_avg,
    g.global_stddev,
    (t.avg_reading - g.global_avg) / NULLIF(g.global_stddev, 0) AS z_score
FROM SCAN_TILE_AGG t
CROSS JOIN global_stats g;

SELECT * FROM FEATURE_TILE_ZSCORE
ORDER BY z_score DESC NULLS LAST
LIMIT 20;

## Combined Feature View

Pull all three features together alongside the reference-range status so the tile-level
output is a single, analysis-ready table.

In [0]:
%%sql -r dataframe_13
CREATE OR REPLACE VIEW SCAN_TILE_FEATURES AS
SELECT
    s.tile,
    s.n_readings,
    s.avg_reading,
    s.max_reading,
    s.avg_status,
    s.max_status,
    h.hotspot_flag,
    v.variability_index,
    z.z_score
FROM SCAN_TILE_STATUS s
LEFT JOIN FEATURE_HOTSPOT_FLAG     h USING (tile)
LEFT JOIN FEATURE_VARIABILITY_INDEX v USING (tile)
LEFT JOIN FEATURE_TILE_ZSCORE       z USING (tile);

SELECT * FROM SCAN_TILE_FEATURES
ORDER BY z_score DESC NULLS LAST;